# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarahnjunge/starter-notebooks/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Sarahnjunge/starter-notebooks.git
%cd starter-notebooks
!ls data/raw

Cloning into 'starter-notebooks'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 173 (delta 75), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 1.90 MiB | 11.56 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/starter-notebooks
content_refresh_anonymized.csv


## 1. Distributions


Before testing individual signals, I first look at the distributions of the main numeric fields. Several traffic and engagement variables are heavily right-skewed, so a small number of pages have much larger values than most pages.

This matters because raw averages can be misleading for heavy-tailed fields. For the signal tests, I will therefore use medians and quantile-based groups where appropriate rather than assuming the variables are normally distributed.


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


In [4]:
# Inspect distributions of key numeric fields

key_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate"
]

distribution_summary = df[key_fields].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
).T

distribution_summary["skew_ratio_p99_median"] = (
    distribution_summary["99%"] /
    distribution_summary["50%"].replace(0, np.nan)
)

print(distribution_summary.round(3).to_string())

                          count      mean        std   min    25%     50%      75%       90%        99%       max  skew_ratio_p99_median
impressions_90d         30000.0  5200.366  16838.020   1.0   81.0  731.00  3615.25  12136.40  73505.830  517715.0                100.555
clicks_90d              30000.0    16.097     75.077   0.0    0.0    1.00     7.00     32.00    253.010    4178.0                253.010
sessions_90d            30000.0    37.067    107.069   1.0    2.0    7.00    27.00     88.00    451.010    4345.0                 64.430
impressions_last_30d    30000.0  1429.059   5643.852   0.0   10.0  139.00   768.00   2991.10  22267.050  238796.0                160.195
impressions_prev_30d    30000.0  1783.078   6150.430   0.0   19.0  210.00  1143.00   4065.00  26293.930  218786.0                125.209
content_age_days        30000.0   256.168    132.708  90.0  132.0  236.00   333.00    463.00    537.000     564.0                  2.275
days_since_last_update  30000.0    46.098

## 1. Distributions

The traffic and engagement fields are heavily right-skewed. For example, `impressions_90d` has a median of 731 but a 99th percentile of about 73,506, while `impressions_last_30d` has a median of 139 and a 99th percentile of about 22,267.

This means a small number of pages have extremely large traffic values, so averages can be misleading. I will use medians or quantile-based groups for the signal tests rather than relying on raw means.


## 2. Signal test #1 / #2 / #3 (verdict each)

### Signal test #1 — Recency of content updates

**Verdict: MIXED**

The down rate increases from 51.1% for pages updated within 30 days to 61.1% for pages updated 91–180 days ago. However, the 181+ day group falls to 47.1%, and it contains only 174 pages, so the relationship is not consistently increasing. This provides some evidence for the signal but not enough to treat update age as a reliable standalone predictor.



In [6]:
# Signal test #1: days since last update

update_signal_test = df.copy()

# Create simple update-recency groups
update_signal_test["update_group"] = pd.cut(
    update_signal_test["days_since_last_update"],
    bins=[0, 30, 90, 180, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"],
    include_lowest=True
)

# Measure the observed down rate in each group
update_results = (
    update_signal_test
    .groupby("update_group", observed=False)
    .agg(
        pages=("content_id", "count"),
        down_rate=("trend_direction", lambda x: (x == "down").mean()),
        median_days_since_update=("days_since_last_update", "median")
    )
    .reset_index()
)

print(update_results.to_string(index=False))

update_group  pages  down_rate  median_days_since_update
   0-30 days  20480   0.511377                      20.0
  31-90 days    175   0.588571                      41.0
 91-180 days   9171   0.611057                     104.0
   181+ days    174   0.471264                     211.0


### Signal test #2 — Click-through rate

**Question:** Are pages with lower CTR more likely to be `down`?

Because CTR is heavily skewed and has many zero values, I will compare the observed down rate across CTR groups rather than relying on the mean. If pages with lower CTR consistently have higher down rates, the signal will be **CONFIRMED**.


In [8]:
# Signal test #2: CTR

ctr_signal_test = df.copy()

# Create interpretable CTR groups
ctr_signal_test["ctr_group"] = pd.cut(
    ctr_signal_test["ctr"],
    bins=[-np.inf, 0, 0.05, 0.20, np.inf],
    labels=[
        "Zero CTR",
        "Very low CTR",
        "Low-to-moderate CTR",
        "Higher CTR"
    ]
)

ctr_results = (
    ctr_signal_test
    .groupby("ctr_group", observed=False)
    .agg(
        pages=("content_id", "count"),
        down_rate=("trend_direction", lambda x: (x == "down").mean()),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print(ctr_results.to_string(index=False))

          ctr_group  pages  down_rate  median_ctr
           Zero CTR  13212   0.496670        0.00
       Very low CTR   1207   0.718310        0.04
Low-to-moderate CTR   5886   0.623853        0.13
         Higher CTR   9695   0.532336        0.44


### Signal test #2 — Click-through rate

**Verdict: MIXED**

Pages with very low CTR have the highest observed down rate at 71.8%, compared with 62.4% for low-to-moderate CTR and 53.2% for higher CTR. However, pages with zero CTR have a lower down rate of 49.7%, so the relationship is not consistently monotonic. CTR therefore provides useful signal, but a simple "lower CTR means more likely down" rule is too strong.


### Signal test #3 — Recent visibility change

**Question:** Are pages whose recent impressions are lower than their previous-period impressions more likely to be `down`?

I will compare the observed down rate across groups based on the ratio of `impressions_last_30d` to `impressions_prev_30d`. A higher down rate among pages with a declining ratio would support the idea that recent visibility loss is a useful signal.


In [9]:
# Signal test #3: recent visibility change

visibility_signal_test = df.copy()

# Calculate recent-to-previous impression ratio
visibility_signal_test["impression_ratio"] = np.where(
    visibility_signal_test["impressions_prev_30d"] > 0,
    visibility_signal_test["impressions_last_30d"] /
    visibility_signal_test["impressions_prev_30d"],
    np.nan
)

# Create interpretable visibility-change groups
visibility_signal_test["visibility_group"] = pd.cut(
    visibility_signal_test["impression_ratio"],
    bins=[-np.inf, 0.8, 1.0, 1.2, np.inf],
    labels=[
        "Large decline (<0.8x)",
        "Small decline (0.8-1.0x)",
        "Small increase (1.0-1.2x)",
        "Large increase (>1.2x)"
    ]
)

visibility_results = (
    visibility_signal_test
    .groupby("visibility_group", observed=False)
    .agg(
        pages=("content_id", "count"),
        down_rate=("trend_direction", lambda x: (x == "down").mean()),
        median_ratio=("impression_ratio", "median")
    )
    .reset_index()
)

print(visibility_results.to_string(index=False))

         visibility_group  pages  down_rate  median_ratio
    Large decline (<0.8x)  16305   0.997363      0.444444
 Small decline (0.8-1.0x)   3853   0.000000      0.897959
Small increase (1.0-1.2x)   2066   0.000000      1.090909
   Large increase (>1.2x)   4388   0.000000      1.625335


### Signal test #3 — Recent visibility change

**Verdict: FALSE**

The recent-to-previous impression ratio appears extremely predictive: pages below 0.8× have a 99.7% observed down rate, while all other groups have a 0% down rate. However, this result is not a valid independent signal because the starter dataset's `trend_direction` target is derived from `trend_pct`, which represents the same underlying change in performance. The near-perfect separation is therefore evidence of target construction rather than a safe predictive relationship.


## 3. The flag-linked test

The baseline includes a stale-content signal based on `days_since_last_update`. This reflects a common content-review assumption: pages that have not been updated for a long time may be more likely to require attention.

I will test whether a stale-content threshold actually separates pages with different observed down rates. The goal is not to prove that stale content causes decline, but to check whether the flag has empirical support in this dataset.


In [10]:
# Flag-linked test: stale content

flag_test = df.copy()

flag_test["stale_flag"] = (
    flag_test["days_since_last_update"] > 180
)

flag_results = (
    flag_test
    .groupby("stale_flag")
    .agg(
        pages=("content_id", "count"),
        down_rate=("trend_direction", lambda x: (x == "down").mean()),
        median_days_since_update=("days_since_last_update", "median")
    )
    .reset_index()
)

print(flag_results.to_string(index=False))

 stale_flag  pages  down_rate  median_days_since_update
      False  29826   0.542480                      20.0
       True    174   0.471264                     211.0


## 4. What this means in practice

The content team should treat update recency and CTR as supporting signals rather than standalone reasons to act, because both showed mixed relationships with observed decline. The stale-content flag is not supported by this dataset, while the recent-impression-change signal is too closely tied to the target to use as an independent predictor. A better review system should combine multiple independent signals and validate them against unseen outcomes before recommending action.


In [11]:
# Summarize the signal-test verdicts

signal_verdicts = {
    "Update recency": "MIXED",
    "CTR": "MIXED",
    "Recent impression change": "FALSE",
    "Stale-content flag": "OPPOSITE"
}

for signal, verdict in signal_verdicts.items():
    print(f"{signal}: {verdict}")


Update recency: MIXED
CTR: MIXED
Recent impression change: FALSE
Stale-content flag: OPPOSITE


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.